# 🧪 Getting Started with the Open Source Synthetic Data SDK

🎥 [Watch the video](https://www.youtube.com/watch?v=jM-fkYC8_LA)

Turn a real, sensitive table into a **privacy-safe synthetic copy** that looks and behaves the
same — but contains no real people. Two steps: **train** a model on your data, then **generate**
as much synthetic data as you want from it.

Built on the Apache-2.0 [Mostly AI SDK](https://github.com/mostly-ai/mostlyai). We'll use the US
Census Income dataset (~50k rows).


## 1. Install

It's a normal Python library. Install it, then restart Python.


In [ ]:
%pip install -U 'mostlyai[local]'
dbutils.library.restartPython()


## 2. Start the SDK

`local` mode runs everything on your own Databricks compute.


In [ ]:
from mostlyai.sdk import MostlyAI

mostly = MostlyAI()


## 3. Load your real data

Any table works — CSV, Parquet, or a Unity Catalog table. Here, US Census Income.


In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/mostly-ai/public-demo-data/dev/census/census.csv.gz')

# Or from Unity Catalog:  df = spark.read.table('main.default.customers').toPandas()
df.head()


## 4. Train a generator

The SDK learns the patterns in your data and returns a *generator*. We cap training at 1 minute
for the demo — drop `max_training_time` for best quality.


In [ ]:
g = mostly.train(config={
    'name': 'US Census Income',
    'tables': [{
        'name': 'census',
        'data': df,
        'tabular_model_configuration': {'max_training_time': 1},
    }],
})


## 5. Check the quality report

Is it accurate? Is it private? The report answers both — accuracy score, distribution charts
(real vs. synthetic), and privacy metrics.


In [ ]:
g.reports(display=True)


## 6. Sample on the fly

`probe` draws fresh synthetic rows instantly — nothing is saved. Great for a quick look.


In [ ]:
mostly.probe(g, size=100_000).head()


## 7. Shape the data to your needs

Fix some columns and let the model fill in the rest. Here: only 35-year-old males.


In [ ]:
seed = pd.DataFrame({'age': [35] * 10_000, 'sex': ['Male'] * 10_000})
mostly.probe(g, seed=seed).head()


## 8. Generate and save to Unity Catalog

Create a synthetic dataset and write it back as a governed table. No real data inside, so you
can share it freely.


In [ ]:
synthetic = mostly.generate(g, size=10_000).data()

(spark.createDataFrame(synthetic)
      .write.mode('overwrite')
      .saveAsTable('main.default.census_income_synthetic'))

synthetic.head()


## That's it

`train` → `probe` / `generate`. Open source, privacy-safe, and ready to share.

**Going further:** add a `differential_privacy` block for a formal privacy guarantee, train
multi-table or time-series generators, or synthesize text columns. See the
[docs](https://mostly-ai.github.io/mostlyai/usage/).
